# C06 组合优化与课程收官

本节目标：将策略信号映射为可执行权重，并与基准比较。


## 流程
1. 估计均值与协方差
2. 求解夏普最大化权重
3. 滚动回测对比等权基准


In [ ]:
MODE = "offline"
START_DATE = "2018-01-31"
END_DATE = "2022-12-31"
N_ASSETS = 6
SEED = 42

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(SEED)


In [ ]:
# 生成月度收益样本（离线）
dates = pd.date_range(START_DATE, END_DATE, freq="ME")
assets = [f"ETF_{i}" for i in range(1, N_ASSETS + 1)]

mu_true = np.linspace(0.004, 0.011, N_ASSETS)
A = np.random.normal(size=(N_ASSETS, N_ASSETS))
cov_true = (A @ A.T) * 0.0008

ret_df = pd.DataFrame(
    np.random.multivariate_normal(mu_true, cov_true, size=len(dates)),
    index=dates,
    columns=assets,
)
ret_df.head()


## 夏普最大化求解
- 目标：最大化风险调整后收益
- 约束：长仓、权重和为 1
- 无 scipy 时回退到波动率倒数加权


In [ ]:
try:
    from scipy.optimize import minimize
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False


def optimize_sharpe(mean_ret, cov_mat, rf=0.0):
    n = len(mean_ret)
    if not HAS_SCIPY:
        # 离线兜底：简化风险平价近似
        inv_vol = 1.0 / np.sqrt(np.diag(cov_mat))
        return inv_vol / inv_vol.sum()

    def objective(w):
        port_ret = w @ mean_ret
        port_vol = np.sqrt(w @ cov_mat @ w)
        return -(port_ret - rf) / (port_vol + 1e-12)

    x0 = np.ones(n) / n
    bounds = [(0.0, 1.0)] * n
    constraints = [{"type": "eq", "fun": lambda w: w.sum() - 1.0}]

    res = minimize(objective, x0=x0, bounds=bounds, constraints=constraints)
    return res.x

w_opt = optimize_sharpe(ret_df.mean().values, ret_df.cov().values)
w_eq = np.ones(N_ASSETS) / N_ASSETS
weights = pd.DataFrame({"asset": assets, "w_opt": w_opt, "w_eq": w_eq}).round(4)
weights


## 滚动优化回测
每月执行：
- 用过去 12 个月估计参数
- 计算目标权重
- 在下月收益上验证


In [ ]:
lookback = 12
dates_bt, r_opt, r_eq = [], [], []

for t in range(lookback, len(ret_df) - 1):
    hist = ret_df.iloc[t - lookback:t]
    nxt = ret_df.iloc[t + 1]
    w = optimize_sharpe(hist.mean().values, hist.cov().values)

    dates_bt.append(ret_df.index[t + 1])
    r_opt.append(float(w @ nxt.values))
    r_eq.append(float(nxt.mean()))

bt = pd.DataFrame({"optimized": r_opt, "equal_weight": r_eq}, index=dates_bt)
nav = (1 + bt).cumprod()

fig, ax = plt.subplots(figsize=(10, 4))
nav.plot(ax=ax, title="C06 组合优化 vs 等权基准")
ax.set_ylabel("NAV")
plt.show()


def perf(r):
    ann_ret = r.mean() * 12
    ann_vol = r.std() * np.sqrt(12)
    sharpe = ann_ret / (ann_vol + 1e-12)
    mdd = ((1 + r).cumprod() / (1 + r).cumprod().cummax() - 1).min()
    return ann_ret, ann_vol, sharpe, mdd

summary = pd.DataFrame(
    [perf(bt["optimized"]), perf(bt["equal_weight"])],
    columns=["annual_return", "annual_vol", "sharpe", "max_drawdown"],
    index=["optimized", "equal_weight"],
).round(4)
summary


## 课程收官
你已完成完整链路：
数据 -> 因子 -> 回测 -> 组合优化

课后练习：加入“单资产权重上限 35%”约束并比较结果。
